## Heatmap creation
Create neuron*time matrix and heatmap from ROI data extracted by minian


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re 
from scipy.cluster import hierarchy
from scipy import cluster   
from scipy.stats import zscore
from scipy.signal import find_peaks
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import shutil
%matplotlib inline

In [ ]:

# down-sampling methods
def downsample_data_30(df):
    df_sorted = df.sort_values(['unit_id', 't']).copy()
    df_sorted['group'] = df_sorted.groupby('unit_id')['t'].transform(lambda x: (x - x.min()) // 3)
    df_downsampled = df_sorted.groupby(['unit_id', 'group']).agg({'t': 'first', 'value': 'mean'}).reset_index(level='unit_id').reset_index(drop=True)
    return df_downsampled

def downsample_data_20(df):
    df_sorted = df.sort_values(['unit_id', 't']).copy()
    df_sorted['group'] = df_sorted.groupby('unit_id')['t'].transform(lambda x: (x - x.min()) // 2)
    df_downsampled = df_sorted.groupby(['unit_id', 'group']).agg({'t': 'first', 'value': 'mean'}).reset_index(level='unit_id').reset_index(drop=True)
    return df_downsampled

In [ ]:

def get_processed_data(path_signal_TST, path_signal_OFT, path_signal_aligned):

    df_signal_TST = pd.read_csv(path_signal_TST+'/C.csv')
    df_raw_TST = pd.read_csv(path_signal_TST+'/YrA.csv')
    if os.path.exists(path_signal_OFT+'/C.csv'):
        df_signal_OFT = pd.read_csv(path_signal_OFT+'/C.csv')
        df_raw_OFT = pd.read_csv(path_signal_OFT+'/YrA.csv')
    else:
        print('Warning: OFT data not found.')
        df_signal_OFT = None
        df_raw_OFT = None

    #check if aligment data exists
    if not os.path.exists(path_signal_aligned):
        df_signal_aligned = None
    else:
        df_signal_aligned = pd.read_csv(path_signal_aligned)

    
    #checkif drop data exist
    try:
        df_drop_id_TST = pd.read_excel(path_signal_TST+'/pic/drop points.xlsx',header=None) 
    except:
        try:
            df_drop_id_TST = pd.read_excel(path_signal_TST+'/pic/drop.xlsx',header=None)
        except:
            df_drop_id_TST = None
            
    try:
        df_drop_id_OFT = pd.read_excel(path_signal_OFT+'/pic/drop points.xlsx',header=None) 
    except:
        try:
            df_drop_id_OFT = pd.read_excel(path_signal_OFT+'/pic/drop.xlsx',header=None)
        except:
            df_drop_id_OFT = None

    # Ensure consistent column names
    df_signal_TST = df_signal_TST.rename(columns={"unit_id":"unit_id","frame":"t","C":"value"})
    df_raw_TST = df_raw_TST.rename(columns={"unit_id":"unit_id","frame":"t","YrA":"value"})
    if df_signal_OFT is not None:
        df_signal_OFT = df_signal_OFT.rename(columns={"unit_id":"unit_id","frame":"t","C":"value"})
    if df_raw_OFT is not None:
        df_raw_OFT = df_raw_OFT.rename(columns={"unit_id":"unit_id","frame":"t","YrA":"value"})
    print(f'PREVIEW {df_signal_TST.iloc[:,:5]}')

    if df_drop_id_TST is not None:
        valid_ids_TST = set(df_signal_TST['unit_id'].unique()) - set(df_drop_id_TST[0].values.tolist())
    else:
        valid_ids_TST = set(df_signal_TST['unit_id'].unique())

    if df_drop_id_OFT is not None and df_signal_OFT is not None:
        valid_ids_OFT = set(df_signal_OFT['unit_id'].unique()) - set(df_drop_id_OFT[0].values.tolist())
    else:
        if df_signal_OFT is not None:
            valid_ids_OFT = set(df_signal_OFT['unit_id'].unique())
        else:
            valid_ids_OFT = None

# Step 3: Filter valid IDs in raw (YrA) and signal (C) data getting from CNMF results
    df_raw_TST = df_raw_TST[df_raw_TST['unit_id'].isin(valid_ids_TST)].copy()
    df_signal_TST = df_signal_TST[df_signal_TST['unit_id'].isin(valid_ids_TST)].copy()
    if df_raw_OFT is not None: 
        df_raw_OFT = df_raw_OFT[df_raw_OFT['unit_id'].isin(valid_ids_OFT)].copy()
    if df_signal_OFT is not None:
        df_signal_OFT = df_signal_OFT[df_signal_OFT['unit_id'].isin(valid_ids_OFT)].copy()

# Step 4: getting sufficient valid IDs in aligned data
    if df_signal_aligned is not None:

        if df_drop_id_TST is not None:
            valid_ids_aligned_TST = set(df_signal_aligned['TST_Unit_ID'].unique()) - set(df_drop_id_TST[0].values.tolist()) 
        else:
            valid_ids_aligned_TST = set(df_signal_aligned['TST_Unit_ID'].unique())

        if df_drop_id_OFT is not None:
            valid_ids_aligned_OFT = set(df_signal_aligned['OFT_Unit_ID'].unique()) - set(df_drop_id_OFT[0].values.tolist())
        else:
            if df_signal_OFT is not None:
                valid_ids_aligned_OFT = set(df_signal_aligned['OFT_Unit_ID'].unique())
            
        df_signal_aligned = df_signal_aligned[df_signal_aligned['TST_Unit_ID'].isin(valid_ids_aligned_TST) & df_signal_aligned['OFT_Unit_ID'].isin(valid_ids_aligned_OFT)].reset_index(drop=True)
    else:
        valid_ids_aligned_TST,valid_ids_aligned_OFT = None,None

    return df_raw_TST,df_signal_TST,df_raw_OFT,df_signal_OFT,df_signal_aligned


In [ ]:
#Warpper and visualization for down-sampling
def downsampling(df_raw_TST,df_signal_TST,df_signal_OFT,test):
        #test1 20 hz to 10 hz; test2 30 hz to 10 hz
        if test == 'test1':
            print(f'Detected {test} use 20_10 sampling')
            df_raw_TST = downsample_data_20(df_raw_TST)
            df_signal_TST_before = df_signal_TST.copy()
            df_signal_TST = downsample_data_20(df_signal_TST)
            if df_signal_OFT is not None:
                df_signal_OFT_before = df_signal_OFT.copy()
                df_signal_OFT = downsample_data_20(df_signal_OFT)
        if test == 'test2':
            print(f'Detected {test} use 30_10 sampling')
            df_raw_TST = downsample_data_30(df_raw_TST)
            df_signal_TST_before = df_signal_TST.copy()
            df_signal_TST = downsample_data_30(df_signal_TST)
            if df_signal_OFT is not None:
                df_signal_OFT_before = df_signal_OFT.copy()
                df_signal_OFT = downsample_data_30(df_signal_OFT)
        print(f'Check downsampled size: {df_signal_TST.groupby("unit_id").size().min()}')

        #visualize down_sample effect
        plt.figure(figsize=(15, 6))

        
        plt.subplot(2, 1, 1)
        before_data = df_signal_TST_before[df_signal_TST_before['unit_id'] == df_signal_TST["unit_id"].values.min()]
        plt.plot(before_data['t'], before_data['value'], 'b-', linewidth=0.8, label='before')
        plt.title('before (unit_id=0)')
        plt.xlabel('frame')
        plt.ylabel('signal')

        plt.grid(alpha=0.3)
        plt.legend()

        
        plt.subplot(2, 1, 2)
        after_data = df_signal_TST[df_signal_TST['unit_id'] == df_signal_TST["unit_id"].values.min()]
        plt.plot(after_data.index, after_data['value'], 'r-', linewidth=0.8, label='after')
        plt.title('after (unit_id=0)')
        plt.xlabel('frame')
        plt.ylabel('signal')
        
        plt.grid(alpha=0.3)
        plt.legend()

        
        plt.tight_layout()
        plt.show()
        plt.close()
        return df_raw_TST,df_signal_TST,df_signal_OFT

In [ ]:

def resized_zscore_df(df_raw_TST, df_signal_TST, df_raw_OFT, df_signal_OFT):
    df_TST = pd.merge(df_raw_TST, df_signal_TST, on=['unit_id', 't'], suffixes=('_YrA', '_C'))
    if df_signal_OFT is None or df_raw_OFT is None:
        df_OFT = None
    else:
        df_OFT = pd.merge(df_raw_OFT, df_signal_OFT, on=['unit_id', 't'], suffixes=('_YrA', '_C'))

    df_TST['residual'] = df_TST['value_YrA'] - df_TST['value_C']
    df_TST['value_z'] = df_TST['value_C'] / df_TST.groupby('unit_id')['residual'].transform('std')

    if df_signal_OFT is not None and df_raw_OFT is not None:
        df_OFT['residual'] = df_OFT['value_YrA'] - df_OFT['value_C']
        df_OFT['value_z'] = df_OFT['value_C'] / df_OFT.groupby('unit_id')['residual'].transform('std')
    resized_df_signal_TST = df_TST.pivot(index='t', columns='unit_id', values='value_z').reset_index(drop=True) 
    if df_OFT is not None:
        resized_df_signal_OFT = df_OFT.pivot(index='t', columns='unit_id', values='value_z').reset_index(drop=True)
    else:
        resized_df_signal_OFT = None  

    # z-score normalization
    resized_df_signal_TST = resized_df_signal_TST.apply(zscore, axis=0)
    if resized_df_signal_OFT is not None:
        resized_df_signal_OFT = resized_df_signal_OFT.apply(zscore, axis=0)
    print(f'Time range = {df_TST.groupby("unit_id").size().min()} -- {df_TST.groupby("unit_id").size().max()}')
    return resized_df_signal_TST, resized_df_signal_OFT


In [ ]:
def get_tables(resized_df_signal_TST,resized_df_signal_OFT,df_signal_aligned,test,outpaths):
    if test == 'test2':
        if df_signal_aligned is  None:
            aligned_table_TST = None
            aligned_table_OFT = None
            not_aligned_table_TST = resized_df_signal_TST.copy()
            if resized_df_signal_OFT is not None:
                not_aligned_table_OFT = resized_df_signal_OFT.copy()
            else:
                not_aligned_table_OFT = None
        else:
            target_columns_TST = df_signal_aligned['TST_Unit_ID'].tolist() 

            aligned_table_TST = resized_df_signal_TST[target_columns_TST].copy()  
            aligned_table_TST.columns = [f'TST_{col}' for col in aligned_table_TST.columns]  

            
            not_aligned_id_TST = set(resized_df_signal_TST.columns) - set(df_signal_aligned['TST_Unit_ID'].unique())
            not_aligned_table_TST = resized_df_signal_TST[list(not_aligned_id_TST)].copy()
            not_aligned_table_TST.columns = [f'TST_{col}' for col in not_aligned_table_TST.columns]

            resized_df_signal_TST.columns = [f'TST_{col}' for col in resized_df_signal_TST.columns]  


            if resized_df_signal_OFT is not None:
                
                target_columns_OFT = df_signal_aligned['OFT_Unit_ID'].tolist()
 
                aligned_table_OFT = resized_df_signal_OFT[target_columns_OFT].copy()  
                aligned_table_OFT.columns = [f'OFT_{col}' for col in aligned_table_OFT.columns] 

                not_aligned_id_OFT = set(resized_df_signal_OFT.columns) - set(df_signal_aligned['OFT_Unit_ID'].unique())
                not_aligned_table_OFT = resized_df_signal_OFT[list(not_aligned_id_OFT)].copy()
                not_aligned_table_OFT.columns = [f'OFT_{col}' for col in not_aligned_table_OFT.columns] 

                resized_df_signal_OFT.columns = [f'OFT_{col}' for col in resized_df_signal_OFT.columns]  
            else:
                aligned_table_OFT = None
                not_aligned_table_OFT = None

    elif test == 'test1':
        not_aligned_table_TST = resized_df_signal_TST.copy()
        not_aligned_table_TST.columns = [f'TST_{col}' for col in not_aligned_table_TST.columns]
    

    if test == 'test1':
        not_aligned_table_TST.to_csv(outpaths[2], index=False)

    elif test == 'test2':

        if aligned_table_TST is not None:
            aligned_table_TST.to_csv(outpaths[0], index=False)

        if aligned_table_OFT is not None:
            aligned_table_OFT.to_csv(outpaths[2], index=False)

        not_aligned_table_TST.to_csv(outpaths[1], index=False)

        try:
            not_aligned_table_OFT.to_csv(outpaths[3], index=False)
        except Exception as e:
            print(f"Error saving not_aligned_table_OFT: {e}")

        resized_df_signal_TST.to_csv(outpaths[4], index=False)
        
        try:
            if resized_df_signal_OFT:
                resized_df_signal_OFT.to_csv(outpaths[5], index=False)
        except Exception as e:
            print(f"Error saving resized_df_signal_OFT: {e}")
            
    return aligned_table_TST,not_aligned_table_TST,aligned_table_OFT,not_aligned_table_OFT


In [2]:
def signal_matrics(resized_df_signal_TST,resized_df_signal_OFT,outpaths):
    df_signal_total_TST = pd.DataFrame(columns=['TST_Unit_ID', 'TST_freq', 'TST_max_peak'])
    df_signal_total_OFT = pd.DataFrame(columns=['OFT_Unit_ID', 'OFT_freq', 'OFT_max_peak'])
    valid_ids_TST = list(resized_df_signal_TST.columns)
    valid_ids_OFT = list(resized_df_signal_OFT.columns)

    for i in range(len(valid_ids_OFT)):
        OFT_Unit_ID = valid_ids_OFT[i]
        cell_OFT = resized_df_signal_OFT[OFT_Unit_ID]

        # 对cell_OFT进行峰值检测
        peaks_OFT, properties_OFT = find_peaks(
        cell_OFT,
        height=1.5,  # 峰值最小高度，根据数据特征调整
        # distance=1,  # 峰值间最小距离（单位：数据点）
        prominence=0.5  # 峰值突出度，用于过滤小波动
        )

        # 将cell_OFT的频率
        freq_OFT = len(peaks_OFT) / 5
        # print(f'ID = {OFT_Unit_ID} cell_OFT 的频率为{freq_OFT}')
        # 将cell_OFT的最大峰值
        max_peak_OFT = max(properties_OFT['peak_heights'])
        # print(f'ID = {OFT_Unit_ID} cell_OFT 的最大峰值为{max_peak_OFT}')
        # 将检测到的cell_OFT的频率和最大峰值保存到df_signal_total_OFT中
        df_signal_total_OFT.loc[i,'OFT_Unit_ID'] = OFT_Unit_ID
        df_signal_total_OFT.loc[i,'OFT_freq'] = freq_OFT
        df_signal_total_OFT.loc[i,'OFT_max_peak'] = max_peak_OFT

    df_signal_total_OFT.to_csv(outpaths[-1], index=False)

    for i in range(len(valid_ids_TST)):
        TST_Unit_ID = valid_ids_TST[i]
        cell_TST = resized_df_signal_TST[TST_Unit_ID]

        # 对cell_TST进行峰值检测
        peaks_TST, properties_TST = find_peaks(
        cell_TST,
        height=1.5,  # 峰值最小高度，根据数据特征调整
        # distance=1,  # 峰值间最小距离（单位：数据点）
        prominence=0.5  # 峰值突出度，用于过滤小波动
        )

        # 将cell_TST的频率
        freq_TST = len(peaks_TST) / 5
        print(f'ID = {TST_Unit_ID} cell_TST 的频率为{freq_TST}')
        # 将cell_TST的最大峰值
        max_peak_TST = max(properties_TST['peak_heights'])
        print(f'ID = {TST_Unit_ID} cell_TST 的最大峰值为{max_peak_TST}')
        # 将检测到的cell_TST的频率和最大峰值保存到df_signal_total_TST中
        df_signal_total_TST.loc[i,'TST_Unit_ID'] = TST_Unit_ID
        df_signal_total_TST.loc[i,'TST_freq'] = freq_TST
        df_signal_total_TST.loc[i,'TST_max_peak'] = max_peak_TST

    df_signal_total_TST.to_csv(outpaths[-2], index=False)
    return df_signal_total_TST, df_signal_total_OFT

In [ ]:

def heatmap_plot(data,subtitle,filepath,is_save=False):
    minmax_scaler = MinMaxScaler()
    minmax_normalized_data = minmax_scaler.fit_transform(data)
    minmax_normalized_df = pd.DataFrame(minmax_normalized_data, columns=data.columns)

    g = sns.clustermap(minmax_normalized_df.T,
                    method='ward',
                    metric='euclidean',
                    cmap='coolwarm',
                    col_cluster=False,
                    row_cluster=False,  
                    figsize=(10, 3), 
                    cbar_kws={'label': 'Normalized Value'})  

    g.figure.suptitle(subtitle, y=1.05)  # 
    if is_save:
        plt.savefig(filepath + '.pdf',
                    format='pdf',
                    bbox_inches='tight',  
                    dpi=300,
                    transparent=True) 
    plt.close()

    g = sns.clustermap(minmax_normalized_df.T,
                    method='ward',
                    metric='euclidean',
                    cmap='coolwarm',
                    col_cluster=False,
                    row_cluster=True,  # row clustering enabled
                    figsize=(6, 3), 
                    cbar_kws={'label': 'Normalized Value'})


    g.figure.suptitle(subtitle + ' clustered', y=1.05) 

    if is_save:
        g.data.to_csv(filepath + '_clustered.csv', index=False)
        plt.savefig(filepath + '_clustered.pdf',
                format='pdf',
                bbox_inches='tight',  
                dpi=300,
                transparent=True)  
    plt.show()
    plt.close()
    
    ax = plt.figure(figsize=(10, 6))
    Z = hierarchy.linkage(minmax_normalized_df.T, method='ward',metric='euclidean')
    hierarchy.dendrogram(Z, labels=data.columns,orientation='left')
    heatmapfig = ax.get_figure()
    plt.gca().axes.get_xaxis().set_visible(False)
    for spine in plt.gca().spines.values():
        spine.set_visible(False)
    plt.gca().invert_yaxis()
    if is_save:
        heatmapfig.savefig(filepath + '_clustered_dendrogram.pdf', dpi=100)
    plt.show()
    plt.close()



#


 -loop from celltypes in test1
 -loop from condition groups in test2

In [ ]:
plt.close('all')#clear plt cache

test1 refers short-term stress experiments test2 refers to long-term stress experiments

In [ ]:
projectpath = r'path'
test = 'test2'

#test1 Params
miceID = range(1,11) #VIP:1~4;CamkII_TST:1~10;SST:1~3;PV:1~7;Hsyn:1~5
cell_type = 'cell=type'#PV

#test2 Params
test2_IDs =['list of micIDs']

condition = 'pre'  #pre,post,rescue


In [ ]:
for mice in test2_IDs:
    if test == 'test1':
        print(('test1 mode Skipping this loop'))
        continue
    
    #数据路径
    print(f'Processing {mice}{condition} data')
    signal_dir = os.path.join(projectpath,'signal_data',test,condition,mice)
    path_signal_TST = os.path.join(signal_dir,'TST')
    path_signal_OFT = os.path.join(signal_dir,'OFT')
    if not os.path.exists(path_signal_TST) or not os.path.exists(path_signal_OFT):
        print(f'Skipping {mice}{condition} as no signal folder found')
        continue
    path_signal_aligned = os.path.join(projectpath,'signal_alignment',condition,mice,'c_trace_peak_stats.csv')

    outdir = os.path.join(signal_dir,'signal_save')
    if os.path.exists(outdir):shutil.rmtree(outdir)
    if not os.path.exists(outdir):os.makedirs(outdir)
    outpath1 = os.path.join(outdir,f'{mice}_{condition}_aligned_table_TST.csv')
    outpath2 = os.path.join(outdir,f'{mice}_{condition}_not_aligned_table_TST.csv')
    outpath3 = os.path.join(outdir,f'{mice}_{condition}_aligned_table_OFT.csv')
    outpath4 = os.path.join(outdir,f'{mice}_{condition}_not_aligned_table_OFT.csv')
    outpath5 = os.path.join(outdir, f'{mice}_{condition}_all_cell_table_TST.csv')
    outpath6 = os.path.join(outdir, f'{mice}_{condition}_all_cell_table_OFT.csv')
    outpath7 = os.path.join(outdir, f'{mice}_{condition}_cell_metrics_TST.csv')
    outpath8 = os.path.join(outdir, f'{mice}_{condition}_cell_metrics_OFT.csv')
    outpaths = [outpath1,outpath2,outpath3,outpath4,outpath5,outpath6,outpath7,outpath8]
    fig_dir = os.path.join(projectpath,'signal_data','firingrate_heatmap',test,condition,mice)
    if not os.path.exists(fig_dir):os.makedirs(fig_dir)

    df_raw_TST,df_signal_TST,df_raw_OFT,df_signal_OFT,df_signal_aligned = get_processed_data(path_signal_TST,path_signal_OFT,path_signal_aligned)#读取数据
    df_raw_TST,df_signal_TST,df_signal_OFT = downsampling(df_raw_TST,df_signal_TST,df_signal_OFT,test)#降维
    resized_df_signal_TST,resized_df_signal_OFT = resized_zscore_df(df_raw_TST,df_signal_TST,df_raw_OFT,df_signal_OFT)#行列转换
    cell_matrics_TST, cell_metrics_OFT = signal_matrics(resized_df_signal_TST,resized_df_signal_OFT,outpaths)#计算频率和峰值
    aligned_table_TST,not_aligned_table_TST,aligned_table_OFT,not_aligned_table_OFT = get_tables(resized_df_signal_TST,resized_df_signal_OFT,df_signal_aligned,test,outpaths)#提取数值
    #热图制作

    if df_signal_aligned is not None:
        heatmap_plot(aligned_table_TST, 'TST aligned heatmap', os.path.join(fig_dir, 'aligned_TST_heatmap'),True)
        if  os.path.exists(path_signal_OFT):
            heatmap_plot(aligned_table_OFT, 'OFT aligned heatmap', os.path.join(fig_dir, 'aligned_OFT_heatmap'),True)
    
    heatmap_plot(resized_df_signal_TST, 'TST heatmap', os.path.join(fig_dir, 'TST_heatmap'),True)

    if resized_df_signal_OFT is not None:
        heatmap_plot(resized_df_signal_OFT, 'OFT heatmap', os.path.join(fig_dir, 'OFT_heatmap'),True)
    else:
        print(f'No OFT data for {mice}{condition}, skipping OFT heatmap plot')
        continue